In [1]:
import pandas as pd

df = pd.read_csv("../data/clean_products.csv")

df.shape

(1666, 24)

In [2]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

c:\Users\BUNNY\OneDrive\Desktop\AI\product-search-ranking\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\BUNNY\OneDrive\Desktop\AI\product-search-ranking\venv\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\BUNNY\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Develop

In [3]:
product_embeddings = model.encode(
    df["weighted_search_text"].tolist(),
    show_progress_bar=True
)

KeyError: 'weighted_search_text'

In [4]:
print(df.columns.tolist())

['id', 'slug', 'title', 'imgs', 'brand', 'category', 'vendor', 'used', 'address', 'availability', 'currency', 'original_price', 'discounted_price', 'specifications', 'description', 'delivery_fee', 'delivery_details', 'warranty', 'warranty_type', 'average_rating', 'num_ratings', 'reviews', 'search_text', 'clean_search_text']


In [6]:
import sys
sys.path.append("..")

from src.preprocessing import clean_text

In [7]:
df["weighted_search_text"] = (
    df["title"].fillna("") + " " +
    df["title"].fillna("") + " " +
    df["title"].fillna("") + " " +
    df["brand"].fillna("") + " " +
    df["category"].fillna("") + " " +
    df["specifications"].fillna("").astype(str) + " " +
    df["description"].fillna("").astype(str)
)

In [8]:
df["weighted_search_text"] = (
    df["weighted_search_text"]
    .astype(str)
    .apply(clean_text)
)

In [9]:
df["weighted_search_text"].head()

0    nothing phone 1 8gb ram 256gb storage non pta ...
1    oppo f21 pro 8gb ram 128gb storage 5g pta appr...
2    tecno spark 10 tecno spark 10 tecno spark 10 t...
3    vivo v27 5g vivo v27 5g vivo v27 5g vivo mobil...
4    apple iphone 15 pro max apple iphone 15 pro ma...
Name: weighted_search_text, dtype: object

In [10]:
product_embeddings = model.encode(
    df["weighted_search_text"].tolist(),
    show_progress_bar=True
)

Batches: 100%|██████████| 53/53 [02:00<00:00,  2.27s/it]


In [11]:
df.to_csv(
    "../data/clean_products_semantic.csv",
    index=False
)

In [12]:
print(df.columns.tolist())

['id', 'slug', 'title', 'imgs', 'brand', 'category', 'vendor', 'used', 'address', 'availability', 'currency', 'original_price', 'discounted_price', 'specifications', 'description', 'delivery_fee', 'delivery_details', 'warranty', 'warranty_type', 'average_rating', 'num_ratings', 'reviews', 'search_text', 'clean_search_text', 'weighted_search_text']


In [13]:
print(df.shape)

print("weighted_search_text" in df.columns)

print(df["weighted_search_text"].head())

(1666, 25)
True
0    nothing phone 1 8gb ram 256gb storage non pta ...
1    oppo f21 pro 8gb ram 128gb storage 5g pta appr...
2    tecno spark 10 tecno spark 10 tecno spark 10 t...
3    vivo v27 5g vivo v27 5g vivo v27 5g vivo mobil...
4    apple iphone 15 pro max apple iphone 15 pro ma...
Name: weighted_search_text, dtype: object


In [14]:
product_embeddings = model.encode(
    df["weighted_search_text"].tolist(),
    show_progress_bar=True
)

Batches: 100%|██████████| 53/53 [01:42<00:00,  1.94s/it]


In [15]:
print("weighted_search_text" in df.columns)

True


In [16]:
df.columns.tolist()

['id',
 'slug',
 'title',
 'imgs',
 'brand',
 'category',
 'vendor',
 'used',
 'address',
 'availability',
 'currency',
 'original_price',
 'discounted_price',
 'specifications',
 'description',
 'delivery_fee',
 'delivery_details',
 'warranty',
 'warranty_type',
 'average_rating',
 'num_ratings',
 'reviews',
 'search_text',
 'clean_search_text',
 'weighted_search_text']

In [17]:
print(product_embeddings.shape)

(1666, 384)


In [18]:
import pickle

with open("../models/product_embeddings.pkl", "wb") as f:
    pickle.dump(product_embeddings, f)

print("saved")

saved


In [19]:
import os

print(
    os.path.getsize(
        "../models/product_embeddings.pkl"
    )
)

2559140


In [20]:
from sklearn.metrics.pairwise import cosine_similarity


def semantic_search(
    query,
    top_n=10
):

    query_embedding = model.encode(
        [query]
    )

    similarity_scores = cosine_similarity(
        query_embedding,
        product_embeddings
    ).flatten()

    top_indices = similarity_scores.argsort()[
        -top_n:
    ][::-1]

    results = df.iloc[top_indices].copy()

    results["semantic_score"] = (
        similarity_scores[top_indices]
    )

    return results[
        [
            "title",
            "brand",
            "category",
            "semantic_score"
        ]
    ]

In [21]:
semantic_search(
    "best apple phone"
)

,title,brand,category,semantic_score
1592,Apple iPhone 13,NaN,Mobile,0.470566
507,"Apple iPad Air 5th Gen 10.9"" M1 Chip 64GB Wi-F...",Apple,Mobile,0.467764
511,"Apple iPad Air 5th Gen 10.9"" M1 Chip 64GB Wi-F...",Apple,Mobile,0.463650
508,"Apple iPad Air 5th Gen 10.9"" M1 Chip 64GB Wi-F...",Apple,Mobile,0.458855
1564,Apple iPhone 11,NaN,Mobile,0.456860
509,"Apple iPad Air 5th Gen 10.9"" M1 Chip 64GB Wi-F...",Apple,Mobile,0.452809
510,"Apple iPad Air 5th Gen 10.9"" M1 Chip 64GB Wi-F...",Apple,Mobile,0.450930
1521,Apple iPhone 14,NaN,Mobile,0.438164
1456,Apple iPhone 14 Pro Max,NaN,Mobile,0.436695
518,Apple iPad Air 5th Gen,Apple,Mobile,0.436611


In [22]:
search_products_v2(
    "best apple phone",
    vectorizer,
    tfidf_matrix,
    df,
    clean_text
)

NameError: name 'search_products_v2' is not defined

In [23]:
import sys
sys.path.append("..")

from src.search_engine import search_products_v2
from src.preprocessing import clean_text

In [24]:
search_products_v2(
    "best apple phone",
    vectorizer,
    tfidf_matrix,
    df,
    clean_text
)

NameError: name 'vectorizer' is not defined

In [25]:
import pickle

In [27]:
with open("../models/vectorizer.pkl", "rb") as f:
    vectorizer = pickle.load(f)

print(type(vectorizer))

<class 'sklearn.feature_extraction.text.TfidfVectorizer'>


In [29]:
with open("../models/tfidf_matrix.pkl", "rb") as f:
    tfidf_matrix = pickle.load(f)

print(tfidf_matrix.shape)

(1666, 4513)


In [31]:
import sys

sys.path.append("..")

from src.search_engine import search_products_v2
from src.preprocessing import clean_text

In [33]:
search_products_v2(
    "best apple phone",
    vectorizer,
    tfidf_matrix,
    df,
    clean_text
)

KeyError: 'rating_score'

In [35]:
print(df.columns.tolist())

['id', 'slug', 'title', 'imgs', 'brand', 'category', 'vendor', 'used', 'address', 'availability', 'currency', 'original_price', 'discounted_price', 'specifications', 'description', 'delivery_fee', 'delivery_details', 'warranty', 'warranty_type', 'average_rating', 'num_ratings', 'reviews', 'search_text', 'clean_search_text', 'weighted_search_text']


In [37]:
import sys

sys.path.append("..")

from src.ranking import create_ranking_features

In [39]:
df = create_ranking_features(df)

In [40]:
df[
    [
        "rating_score",
        "popularity_score"
    ]
].head()

,rating_score,popularity_score
0,0.0,0.0
1,0.0,0.0
2,0.0,0.0
3,0.0,0.0
4,0.0,0.0


In [41]:
search_products_v2(
    "best apple phone",
    vectorizer,
    tfidf_matrix,
    df,
    clean_text
)

[{'title': 'Apple MacBook Pro 16.2" - Apple M2 Max Chip',
  'brand': 'Apple',
  'category': 'Laptop',
  'average_rating': 5.0,
  'num_ratings': 3.0,
  'similarity_score': 0.21167260977814656,
  'final_score': 0.2667850538647173},
 {'title': 'Apple iPhone 13',
  'brand': '',
  'category': 'Mobile',
  'average_rating': 5.0,
  'num_ratings': 6.0,
  'similarity_score': 0.1892519472311442,
  'final_score': 0.24914143296881594},
 {'title': 'Apple iPhone 11',
  'brand': '',
  'category': 'Mobile',
  'average_rating': 5.0,
  'num_ratings': 127.0,
  'similarity_score': 0.1668264176710004,
  'final_score': 0.24212274362924913},
 {'title': 'Apple MacBook Air 13 MGN63 M1 Chip',
  'brand': 'Apple',
  'category': 'Laptop',
  'average_rating': 3.5,
  'num_ratings': 19.0,
  'similarity_score': 0.18481765143371628,
  'final_score': 0.22890610316303145},
 {'title': 'Apple MacBook Air 13.3" MGND3LL/A Gold M1 Chip',
  'brand': 'Apple',
  'category': 'Laptop',
  'average_rating': 4.0,
  'num_ratings': 4.0,

In [42]:
from src.ranking import create_ranking_features

df = create_ranking_features(df)

In [44]:
search_products_v2(
    "best apple phone",
    vectorizer,
    tfidf_matrix,
    df,
    clean_text
)

[{'title': 'Apple MacBook Pro 16.2" - Apple M2 Max Chip',
  'brand': 'Apple',
  'category': 'Laptop',
  'average_rating': 5.0,
  'num_ratings': 3.0,
  'similarity_score': 0.21167260977814656,
  'final_score': 0.2667850538647173},
 {'title': 'Apple iPhone 13',
  'brand': '',
  'category': 'Mobile',
  'average_rating': 5.0,
  'num_ratings': 6.0,
  'similarity_score': 0.1892519472311442,
  'final_score': 0.24914143296881594},
 {'title': 'Apple iPhone 11',
  'brand': '',
  'category': 'Mobile',
  'average_rating': 5.0,
  'num_ratings': 127.0,
  'similarity_score': 0.1668264176710004,
  'final_score': 0.24212274362924913},
 {'title': 'Apple MacBook Air 13 MGN63 M1 Chip',
  'brand': 'Apple',
  'category': 'Laptop',
  'average_rating': 3.5,
  'num_ratings': 19.0,
  'similarity_score': 0.18481765143371628,
  'final_score': 0.22890610316303145},
 {'title': 'Apple MacBook Air 13.3" MGND3LL/A Gold M1 Chip',
  'brand': 'Apple',
  'category': 'Laptop',
  'average_rating': 4.0,
  'num_ratings': 4.0,

In [46]:
semantic_search("gaming laptop")

,title,brand,category,semantic_score
423,HP Victus Gaming Laptop 15-FA1032NE,HP,Laptop,0.675300
419,Hp Victus Gaming Laptop 15-FA1040NE,HP,Laptop,0.661133
397,HP Victus Gaming Laptop 15-FA0031DX,HP,Laptop,0.656347
408,HP Victus 15-FA1093DX Gaming Laptop,HP,Laptop,0.646483
425,MSI Sword 15 A12UE Gaming Laptop,MSI,Laptop,0.642929
638,Lenovo Legion Slim 5 16IRH8 13th Gen Core i7-1...,Lenovo,Laptop,0.630639
404,HP Victus 15-FA0025NR Gaming Laptop,HP,Laptop,0.617593
416,Dell G15 5515 Ryzen Edition Gaming Laptop,Dell,Laptop,0.608801
399,HP Pavilion 15-EG2009NIA Laptop,HP,Laptop,0.607155
379,HP 15S-FQ5295NIA Laptop,HP,Laptop,0.599122


In [48]:
semantic_search("budget samsung mobile")

,title,brand,category,semantic_score
1571,Samsung Galaxy A32,NaN,Mobile,0.613588
1351,Samsung Galaxy A33 5G,NaN,Mobile,0.608919
1576,Samsung Galaxy A22,NaN,Mobile,0.604782
1582,Samsung Galaxy A12,NaN,Mobile,0.601893
1344,Samsung Galaxy A13,NaN,Mobile,0.599219
1353,Samsung Galaxy A53 5G,NaN,Mobile,0.594230
1587,Samsung Galaxy A03,NaN,Mobile,0.594018
1429,Samsung Galaxy S22,NaN,Mobile,0.593337
494,"Samsung Galaxy Tab A8 SM-X205 10.5"" Tablet 64G...",Samsung,Mobile,0.592493
1388,Samsung Galaxy A23,NaN,Mobile,0.587707


In [50]:
semantic_search("wireless earbuds")

,title,brand,category,semantic_score
1037,Tronsmart Apollo Bold True Wireless Earbuds,NaN,Earbuds,0.657858
865,Portable True Wireless Earbuds (YD03),NaN,Earbuds,0.649978
1026,Joyroom TWS True Wireless Earbuds (JR-TL7),NaN,Earbuds,0.642149
1045,Audionic True Wireless Stereo Earbuds,NaN,Earbuds,0.638161
863,itel Wireless Earbuds (T3),NaN,Earbuds,0.632249
988,Joyroom True Wireless Gaming Earbuds (TP1),NaN,Earbuds,0.631884
1025,Joyroom True Wireless Earbuds (JR-TL6),NaN,Earbuds,0.631146
862,itel Wireless Earbuds (KT-01),NaN,Earbuds,0.630182
1125,Joyroom TWS Wireless Earbuds (T13),NaN,Earbuds,0.624145
962,Lenovo X3 TWS Wireless Earbuds,NaN,Earbuds,0.621507


In [52]:
search_products_v2(
    "gaming laptop",
    vectorizer,
    tfidf_matrix,
    df,
    clean_text
)

[{'title': 'HP Victus 15-FA1093DX Gaming Laptop',
  'brand': 'HP',
  'category': 'Laptop',
  'average_rating': 5.0,
  'num_ratings': 1.0,
  'similarity_score': 0.3099222207502379,
  'final_score': 0.3520698512074068},
 {'title': 'HP Victus Gaming Laptop 15-FA0031DX',
  'brand': 'HP',
  'category': 'Laptop',
  'average_rating': 5.0,
  'num_ratings': 1.0,
  'similarity_score': 0.29527634067600916,
  'final_score': 0.33888855914060095},
 {'title': 'M25 Gaming Wireless Earbuds',
  'brand': '',
  'category': 'Earbuds',
  'average_rating': 5.0,
  'num_ratings': 1.0,
  'similarity_score': 0.24630048859160267,
  'final_score': 0.2948102922646351},
 {'title': 'HP Victus 15-FA0025NR Gaming Laptop',
  'brand': 'HP',
  'category': 'Laptop',
  'average_rating': 0.0,
  'num_ratings': 0.0,
  'similarity_score': 0.3119005523799687,
  'final_score': 0.2807104971419719},
 {'title': 'Hp Victus Gaming Laptop 15-FA1040NE',
  'brand': 'HP',
  'category': 'Laptop',
  'average_rating': 0.0,
  'num_ratings': 0

In [54]:
search_products_v2(
    "budget samsung mobile",
    vectorizer,
    tfidf_matrix,
    df,
    clean_text
)

c:\Users\BUNNY\OneDrive\Desktop\AI\product-search-ranking\notebooks\..\src\search_engine.py:55: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  output = output.replace(


[{'title': 'Samsung Galaxy Buds 2',
  'brand': '',
  'category': 'Earbuds',
  'average_rating': 5.0,
  'num_ratings': 83.0,
  'similarity_score': 0.17692941298036335,
  'final_score': 0.24930740572881832},
 {'title': 'Samsung Galaxy Buds 2 Pro',
  'brand': '',
  'category': 'Earbuds',
  'average_rating': 5.0,
  'num_ratings': 37.0,
  'similarity_score': 0.16782675207452608,
  'final_score': 0.23752179535892295},
 {'title': 'Samsung Galaxy Buds Pro',
  'brand': '',
  'category': 'Earbuds',
  'average_rating': 5.0,
  'num_ratings': 32.0,
  'similarity_score': 0.16598793575290902,
  'final_score': 0.23522779582666364},
 {'title': 'Samsung Galaxy A12',
  'brand': '',
  'category': 'Mobile',
  'average_rating': 5.0,
  'num_ratings': 319.0,
  'similarity_score': 0.13790030014790877,
  'final_score': 0.22023989713457542},
 {'title': 'Samsung Galaxy Fit 2',
  'brand': '',
  'category': 'Watch',
  'average_rating': 5.0,
  'num_ratings': 11.0,
  'similarity_score': 0.15310198832897298,
  'final_

In [56]:
search_products_v2(
    "wireless earbuds",
    vectorizer,
    tfidf_matrix,
    df,
    clean_text
)

c:\Users\BUNNY\OneDrive\Desktop\AI\product-search-ranking\notebooks\..\src\search_engine.py:55: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  output = output.replace(


[{'title': 'Soundpeats Mac 2 Wireless Earbuds',
  'brand': '',
  'category': 'Earbuds',
  'average_rating': 5.0,
  'num_ratings': 1.0,
  'similarity_score': 0.39338829172504236,
  'final_score': 0.4271893150847308},
 {'title': 'Mibro Earbuds 2',
  'brand': '',
  'category': 'Earbuds',
  'average_rating': 5.0,
  'num_ratings': 4.0,
  'similarity_score': 0.38405425915802305,
  'final_score': 0.42293934505052216},
 {'title': 'Soundpeats Life ANC Wireless Earbuds',
  'brand': '',
  'category': 'Earbuds',
  'average_rating': 5.0,
  'num_ratings': 1.0,
  'similarity_score': 0.3882066020714193,
  'final_score': 0.4225257943964701},
 {'title': 'Audionic True Wireless Stereo Earbuds',
  'brand': '',
  'category': 'Earbuds',
  'average_rating': 5.0,
  'num_ratings': 2.0,
  'similarity_score': 0.3828308402718805,
  'final_score': 0.41952430476601227},
 {'title': 'M5 Wireless Earbuds',
  'brand': '',
  'category': 'Earbuds',
  'average_rating': 5.0,
  'num_ratings': 2.0,
  'similarity_score': 0.37

In [ ]:
def hybrid_search(
    query,
    top_n=10
):
    pass

In [58]:
query_embedding = model.encode([query])

semantic_scores = cosine_similarity(
    query_embedding,
    product_embeddings
).flatten()

NameError: name 'query' is not defined

In [88]:
from sklearn.metrics.pairwise import cosine_similarity

def hybrid_search(query, top_n=10):

    # Semantic Search
    query_embedding = model.encode([query])

    semantic_scores = cosine_similarity(
        query_embedding,
        product_embeddings
    ).flatten()

    # TF-IDF Search
    query_vector = vectorizer.transform(
        [clean_text(query)]
    )

    tfidf_scores = cosine_similarity(
        query_vector,
        tfidf_matrix
    ).flatten()

    # Combine
    temp_df = df.copy()

    temp_df["semantic_score"] = semantic_scores

    temp_df["tfidf_score"] = tfidf_scores

    if detect_budget_intent(query):

        temp_df["hybrid_score"] = (
            0.55 * temp_df["semantic_score"]
            + 0.20 * temp_df["tfidf_score"]
            + 0.10 * temp_df["rating_score"]
            + 0.05 * temp_df["popularity_score"]
            + 0.10 * temp_df["price_score"]
        )

    else:

        temp_df["hybrid_score"] = (
            0.50 * temp_df["semantic_score"]
            + 0.30 * temp_df["tfidf_score"]
            + 0.15 * temp_df["rating_score"]
            + 0.05 * temp_df["popularity_score"]
        )

    results = temp_df.sort_values(
        "hybrid_score",
        ascending=False
    )

    return results[
        [
            "title",
            "brand",
            "category",
            "discounted_price",
            "price_score",
            "semantic_score",
            "tfidf_score",
            "hybrid_score"
        ]
    ].head(top_n).head(top_n)

In [62]:
hybrid_search("best apple phone")

,title,brand,category,semantic_score,tfidf_score,hybrid_score
1564,Apple iPhone 11,NaN,Mobile,0.456860,0.166826,0.465109
1592,Apple iPhone 13,NaN,Mobile,0.470566,0.189252,0.456750
1456,Apple iPhone 14 Pro Max,NaN,Mobile,0.436695,0.145987,0.426835
1441,Apple iPhone 14 Pro,NaN,Mobile,0.430436,0.143274,0.418666
1521,Apple iPhone 14,NaN,Mobile,0.438164,0.144899,0.417785
1510,Apple iPhone 14 Plus,NaN,Mobile,0.432279,0.141238,0.413744
1344,Samsung Galaxy A13,NaN,Mobile,0.401373,0.049406,0.406564
1582,Samsung Galaxy A12,NaN,Mobile,0.384662,0.049498,0.400730
1571,Samsung Galaxy A32,NaN,Mobile,0.367382,0.045366,0.397301
464,"Apple MacBook Pro 16.2"" - Apple M2 Max Chip",Apple,Laptop,0.344808,0.211673,0.396372


In [63]:
hybrid_search("budget samsung mobile")

,title,brand,category,semantic_score,tfidf_score,hybrid_score
1571,Samsung Galaxy A32,NaN,Mobile,0.613588,0.126388,0.544710
1582,Samsung Galaxy A12,NaN,Mobile,0.601893,0.137900,0.535866
1344,Samsung Galaxy A13,NaN,Mobile,0.599219,0.137642,0.531958
1587,Samsung Galaxy A03,NaN,Mobile,0.594018,0.139392,0.527948
1351,Samsung Galaxy A33 5G,NaN,Mobile,0.608919,0.128232,0.527621
1576,Samsung Galaxy A22,NaN,Mobile,0.604782,0.139662,0.526784
1353,Samsung Galaxy A53 5G,NaN,Mobile,0.594230,0.128922,0.519930
1388,Samsung Galaxy A23,NaN,Mobile,0.587707,0.135388,0.518608
1572,Samsung Galaxy A52s 5G,NaN,Mobile,0.585825,0.114599,0.514437
1382,Samsung Galaxy S22 Ultra,NaN,Mobile,0.582952,0.120990,0.510061


In [65]:
hybrid_search("gaming laptop")

,title,brand,category,semantic_score,tfidf_score,hybrid_score
397,HP Victus Gaming Laptop 15-FA0031DX,HP,Laptop,0.656347,0.295276,0.571989
408,HP Victus 15-FA1093DX Gaming Laptop,HP,Laptop,0.646483,0.309922,0.571451
1637,HP Victus Gaming Laptop Core i7-12650H 16GB 51...,NaN,Laptop,0.543614,0.209203,0.489801
402,HP ProBook 450 G9 Laptop - Intel Core i7-1255U,HP,Laptop,0.523591,0.105861,0.448787
419,Hp Victus Gaming Laptop 15-FA1040NE,HP,Laptop,0.661133,0.305699,0.422276
1647,Lenovo Laptop Ideapad 3 15.6 Inches 11th Gen C...,NaN,Laptop,0.475321,0.094977,0.421387
1640,HP Laptop EQ2180AU 15.6 Inches AMD Ryzen 5 (8G...,NaN,Laptop,0.477289,0.082981,0.418772
423,HP Victus Gaming Laptop 15-FA1032NE,HP,Laptop,0.675300,0.261350,0.416055
1648,HP FQ5099TU 15.6 Inches Core i7 12Gen (8GB RAM...,NaN,Laptop,0.506914,0.023330,0.415689
1636,HP FQ5098TU 15.6 Inches Core i5 12Gen (8GB RAM...,NaN,Laptop,0.498623,0.023349,0.411549


In [66]:
print(type(model))
print(product_embeddings.shape)
print(type(vectorizer))
print(tfidf_matrix.shape)
print(df.shape)

<class 'sentence_transformers.sentence_transformer.model.SentenceTransformer'>
(1666, 384)
<class 'sklearn.feature_extraction.text.TfidfVectorizer'>
(1666, 4513)
(1666, 27)


In [68]:
df["discounted_price"].describe()

count       739.000000
mean      41247.326116
std       96631.352491
min        1149.000000
25%        3899.500000
50%        6599.000000
75%       29299.000000
max      794999.000000
Name: discounted_price, dtype: float64

In [70]:
import numpy as np
from sklearn.preprocessing import MinMaxScaler

In [71]:
df["discounted_price"] = (
    df["discounted_price"]
    .fillna(df["discounted_price"].median())
)

In [73]:
df["log_price"] = np.log1p(
    df["discounted_price"]
)

In [74]:
scaler = MinMaxScaler()

df["price_score"] = scaler.fit_transform(
    df[["log_price"]]
)

In [75]:
df["price_score"] = (
    1 - df["price_score"]
)

In [76]:
df[
    [
        "discounted_price",
        "price_score"
    ]
].sample(10)

,discounted_price,price_score
1207,3099.0,0.848340
1252,6699.0,0.730470
1331,6399.0,0.737476
1251,6499.0,0.735105
746,359999.0,0.121164
1122,1199.0,0.993491
357,6599.0,0.732770
1217,6299.0,0.739884
1472,6599.0,0.732770
1601,6599.0,0.732770


In [77]:
def detect_budget_intent(query):

    budget_keywords = [
        "budget",
        "cheap",
        "affordable",
        "low price",
        "economical",
        "under"
    ]

    query = query.lower()

    return any(
        word in query
        for word in budget_keywords
    )

In [79]:
detect_budget_intent(
    "budget samsung mobile"
)

True

In [80]:
temp_df["hybrid_score"] = (
    0.50 * temp_df["semantic_score"]
    + 0.30 * temp_df["tfidf_score"]
    + 0.15 * temp_df["rating_score"]
    + 0.05 * temp_df["popularity_score"]
)

NameError: name 'temp_df' is not defined

In [85]:
hybrid_search(
    "budget samsung mobile"
)

,title,brand,category,semantic_score,tfidf_score,hybrid_score
1582,Samsung Galaxy A12,NaN,Mobile,0.601893,0.137900,0.575447
1576,Samsung Galaxy A22,NaN,Mobile,0.604782,0.139662,0.566334
1571,Samsung Galaxy A32,NaN,Mobile,0.613588,0.126388,0.548858
1429,Samsung Galaxy S22,NaN,Mobile,0.593337,0.116263,0.545095
1587,Samsung Galaxy A03,NaN,Mobile,0.594018,0.139392,0.541041
1344,Samsung Galaxy A13,NaN,Mobile,0.599219,0.137642,0.539863
1351,Samsung Galaxy A33 5G,NaN,Mobile,0.608919,0.128232,0.527767
1388,Samsung Galaxy A23,NaN,Mobile,0.587707,0.135388,0.521228
1345,Samsung Galaxy A04,NaN,Mobile,0.569136,0.140595,0.520207
1353,Samsung Galaxy A53 5G,NaN,Mobile,0.594230,0.128922,0.516998


In [83]:
[
    "title",
    "discounted_price",
    "price_score",
    "hybrid_score"
]

['title', 'discounted_price', 'price_score', 'hybrid_score']

In [86]:
hybrid_search(
    "cheapest mobile"
)

,title,brand,category,semantic_score,tfidf_score,hybrid_score
1583,me Mobile M-1110 Lite,NaN,Mobile,0.466927,0.245306,0.494847
1582,Samsung Galaxy A12,NaN,Mobile,0.480906,0.049045,0.491133
1598,me Mobile L110,NaN,Mobile,0.473316,0.251069,0.489048
1611,me Mobile L786,NaN,Mobile,0.458272,0.271471,0.484854
1610,me Mobile Power Bold,NaN,Mobile,0.456926,0.271812,0.484182
1576,Samsung Galaxy A22,NaN,Mobile,0.487014,0.049672,0.483564
1626,me mobile L101,NaN,Mobile,0.462307,0.251069,0.482993
1617,me Mobile Power Max,NaN,Mobile,0.434512,0.319187,0.481329
1629,me Mobile L106 Pro,NaN,Mobile,0.456635,0.253442,0.480348
1627,me Mobile L100,NaN,Mobile,0.455540,0.247492,0.478556


In [89]:
hybrid_search("cheapest mobiles")[
    [
        "title",
        "discounted_price",
        "price_score",
        "hybrid_score"
    ]
]

,title,discounted_price,price_score,hybrid_score
1582,Samsung Galaxy A12,6599.0,0.732770,0.484471
1576,Samsung Galaxy A22,6599.0,0.732770,0.478001
1350,Nokia 105 2022,4799.0,0.781473,0.463546
1575,Nokia 110 (2019),6599.0,0.732770,0.452045
1591,Nokia 110 4G,6599.0,0.732770,0.449084
1588,VGO TEL i101,2449.0,0.884329,0.446086
1489,Oppo A16,6599.0,0.732770,0.445993
1581,Dcode Cygnal,6599.0,0.732770,0.444484
1583,me Mobile M-1110 Lite,6599.0,0.732770,0.443421
1372,itel Muzik 400,4299.0,0.798297,0.443318


In [90]:
df[
    df["title"]
    .str.contains(
        "Samsung Galaxy A12",
        case=False,
        na=False
    )
][
    [
        "title",
        "discounted_price",
        "category"
    ]
]

,title,discounted_price,category
1582,Samsung Galaxy A12,6599.0,Mobile


In [91]:
df.sort_values(
    "discounted_price"
)[
    [
        "title",
        "discounted_price",
        "category"
    ]
].head(20)

,title,discounted_price,category
1001,Infinix Sports Bluetooth Earphone (XE07),1149.0,Earbuds
835,Airs Pro TWS Bluetooth Earbuds,1199.0,Earbuds
1122,P47i Bluetooth Wireless Headphones,1199.0,Earbuds
919,M27 TWS Wireless Bluetooth Earbuds,1249.0,Earbuds
833,M20 TWS Wireless Bluetooth Earbuds,1249.0,Earbuds
774,M10 TWS Wireless Bluetooth Earbuds,1249.0,Earbuds
870,Airox Wireless Bluetooth Neckband (NB-01),1299.0,Earbuds
864,Interlink ECO NeckBand Wireless Earbuds,1349.0,Earbuds
975,Coolpad Coolbuds Pro Earbuds,1499.0,Earbuds
843,M90 Pro TWS Gaming Earbuds,1599.0,Earbuds


In [92]:
mobile_prices = (
    df[df["category"] == "Mobile"]
    [["title", "discounted_price"]]
    .sort_values("discounted_price")
)

mobile_prices.head(20)

,title,discounted_price
1588,VGO TEL i101,2449.0
1368,itel Value 100s,2799.0
1380,itel Value 110s,2849.0
1590,itel Power 410,3199.0
1428,VGO TEL i251,3599.0
1535,VGO TEL i261,3749.0
1539,QMobile E400 Pro,3799.0
1412,QMobile SL100 Power,4099.0
1436,VGO TEL S20,4149.0
1372,itel Muzik 400,4299.0


In [94]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1206.53it/s]


In [95]:
import pickle

with open("../models/product_embeddings.pkl", "rb") as f:
    product_embeddings = pickle.load(f)

In [96]:
from src.search_engine import (
    search_products_v5
)

results = search_products_v5(
    "budget samsung mobile",
    vectorizer,
    tfidf_matrix,
    product_embeddings,
    model,
    df,
    clean_text
)

results[:10]

ImportError: cannot import name 'search_products_v5' from 'src.search_engine' (c:\Users\BUNNY\OneDrive\Desktop\AI\product-search-ranking\notebooks\..\src\search_engine.py)

In [98]:
import importlib
import src.search_engine

importlib.reload(src.search_engine)

print(dir(src.search_engine))

['SentenceTransformer', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', '__warningregistry__', 'cosine_similarity', 'get_brands', 'get_semantic_scores', 'np', 'parse_query', 'search_products_v2', 'search_products_v3', 'search_products_v5']


In [99]:
import importlib
import src.search_engine

importlib.reload(src.search_engine)

from src.search_engine import search_products_v5

In [100]:
results = search_products_v5(
    "budget samsung mobile",
    vectorizer,
    tfidf_matrix,
    product_embeddings,
    model,
    df,
    clean_text
)

results[:10]

c:\Users\BUNNY\OneDrive\Desktop\AI\product-search-ranking\notebooks\..\src\search_engine.py:357: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`


[{'title': 'Samsung Galaxy A32',
  'brand': '',
  'category': 'Mobile',
  'average_rating': 5.0,
  'num_ratings': 751.0,
  'semantic_score': 0.6135878562927246,
  'tfidf_score': 0.12638824343068844,
  'final_score': 0.5690703937395997},
 {'title': 'Samsung Galaxy A12',
  'brand': '',
  'category': 'Mobile',
  'average_rating': 5.0,
  'num_ratings': 319.0,
  'semantic_score': 0.6018927097320557,
  'tfidf_score': 0.13790030014790877,
  'final_score': 0.5590654616067638},
 {'title': 'Samsung Galaxy A13',
  'brand': '',
  'category': 'Mobile',
  'average_rating': 5.0,
  'num_ratings': 229.0,
  'semantic_score': 0.5992186665534973,
  'tfidf_score': 0.13764206822938582,
  'final_score': 0.5550369170194547},
 {'title': 'Samsung Galaxy A33 5G',
  'brand': '',
  'category': 'Mobile',
  'average_rating': 5.0,
  'num_ratings': 98.0,
  'semantic_score': 0.6089192628860474,
  'tfidf_score': 0.12823160464105335,
  'final_score': 0.5516554993648648},
 {'title': 'Samsung Galaxy A03',
  'brand': '',
  